# Irrigation Need Prediction — Professional SVM Pipeline

**Task**: Multiclass classification — predict `Irrigation_Need` as **Low**, **Medium**, or **High**.

**Pipeline overview**:
1. Load and inspect data
2. Exploratory Data Analysis (EDA)
3. Feature engineering (interaction & ratio features)
4. Preprocessing with `ColumnTransformer` (OneHotEncoding + StandardScaler)
5. Handle class imbalance (`class_weight='balanced'`)
6. SVM with Stratified K-Fold cross-validation
7. Hyperparameter tuning via `GridSearchCV`
8. Final model evaluation (classification report, confusion matrix, ROC curves)
9. Baseline comparisons (Random Forest, LightGBM)
10. Submission generation

---
## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import (
    LabelEncoder, StandardScaler, OneHotEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, GridSearchCV
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, roc_curve, auc, RocCurveDisplay
)
from sklearn.preprocessing import label_binarize
import lightgbm as lgb

# ---------- Configuration ----------
SEED = 42
np.random.seed(SEED)

# SVM subsampling sizes (RBF kernel is O(n^2)–O(n^3))
SVM_TUNE_SAMPLE  = 100_000   # rows for GridSearchCV
SVM_TRAIN_SAMPLE = 150_000   # rows for final training

print('Configuration set. SEED =', SEED)

---
## 2. Load & Inspect Data

In [ ]:
train_raw = pd.read_csv('train.csv')
test_raw  = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f'Train shape : {train_raw.shape}')
print(f'Test shape  : {test_raw.shape}')
print(f'Submission  : {sample_sub.shape}')
print()
print('--- Train dtypes ---')
print(train_raw.dtypes)
print()
print('--- Missing values (train) ---')
print(train_raw.isnull().sum().sum(), 'total missing')
print()
print('--- Missing values (test) ---')
print(test_raw.isnull().sum().sum(), 'total missing')
print()
train_raw.head()

---
## 3. Exploratory Data Analysis

In [ ]:
TARGET = 'Irrigation_Need'

# --- 3a. Target distribution ---
print('Target distribution:')
target_counts = train_raw[TARGET].value_counts()
print(target_counts)
print()
print('Target proportions:')
print((target_counts / len(train_raw) * 100).round(2), '%')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bar chart
colors = ['#2196F3', '#FF9800', '#F44336']
axes[0].bar(target_counts.index, target_counts.values, color=colors)
axes[0].set_title('Target Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Irrigation Need')
axes[0].set_ylabel('Count')
for i, (label, count) in enumerate(zip(target_counts.index, target_counts.values)):
    pct = count / len(train_raw) * 100
    axes[0].text(i, count + 2000, f'{pct:.1f}%', ha='center', fontsize=10, fontweight='bold')

# Correlation heatmap
numeric_cols = train_raw.select_dtypes(include='number').columns.drop('id')
corr = train_raw[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=axes[1], mask=mask, cmap='coolwarm', center=0,
            linewidths=0.5, annot=True, fmt='.2f', square=True, cbar_kws={'shrink': 0.7})
axes[1].set_title('Feature Correlations', fontsize=13, fontweight='bold')

# Pie chart for imbalance visualization
axes[2].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0, 0, 0.1))
axes[2].set_title('Class Imbalance Visualization', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# --- 3b. Box plots of key numeric features by target ---
key_features = ['Soil_Moisture', 'Temperature_C', 'Humidity',
                'Rainfall_mm', 'Previous_Irrigation_mm', 'Soil_pH']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, feat in zip(axes.ravel(), key_features):
    train_raw.boxplot(column=feat, by=TARGET, ax=ax,
                      patch_artist=True,
                      boxprops=dict(facecolor='lightblue', color='navy'),
                      medianprops=dict(color='red', linewidth=2))
    ax.set_title(feat, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(feat)

plt.suptitle('Numeric Feature Distributions by Irrigation Need', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Feature Engineering

We create domain-relevant interaction and ratio features before preprocessing.
These capture nonlinear relationships that help SVM find better decision boundaries.

In [ ]:
def engineer_features(df):
    """Create domain-relevant interaction and ratio features."""
    df = df.copy()

    # Interaction features
    df['Moisture_x_Temp']     = df['Soil_Moisture'] * df['Temperature_C']
    df['Rainfall_x_Humidity'] = df['Rainfall_mm'] * df['Humidity']
    df['Temp_x_Humidity']     = df['Temperature_C'] * df['Humidity']
    df['Moisture_x_Rainfall'] = df['Soil_Moisture'] * df['Rainfall_mm']

    # Ratio features (with small epsilon to avoid division by zero)
    eps = 1e-6
    df['Irrigation_per_Hectare'] = df['Previous_Irrigation_mm'] / (df['Field_Area_hectare'] + eps)
    df['Rainfall_per_Sunlight']  = df['Rainfall_mm'] / (df['Sunlight_Hours'] + eps)
    df['Moisture_Deficit']       = df['Temperature_C'] - df['Soil_Moisture']

    return df

train_fe = engineer_features(train_raw)
test_fe  = engineer_features(test_raw)

print(f'Train shape after FE: {train_fe.shape}')
print(f'Test shape after FE : {test_fe.shape}')
print()
print('New features:', [c for c in train_fe.columns if c not in train_raw.columns])

---
## 5. Preprocessing with ColumnTransformer

**Why not LabelEncoding for SVM?**  
SVM is a distance-based algorithm — it computes distances in feature space. LabelEncoding maps categories to arbitrary integers (e.g., Loamy=0, Sandy=2), which implies a false ordinal relationship and corrupts distance calculations. **OneHotEncoding** is the correct approach for nominal categorical features with SVM.

We use `ColumnTransformer` to apply:
- `StandardScaler` to numeric features (SVM is sensitive to feature magnitudes)
- `OneHotEncoder` to categorical features (proper encoding for distance-based models)

In [ ]:
# Define column groups
cat_cols = [
    'Soil_Type', 'Crop_Type', 'Crop_Growth_Stage',
    'Season', 'Irrigation_Type', 'Water_Source',
    'Mulching_Used', 'Region'
]

num_cols = [
    'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
    'Electrical_Conductivity', 'Temperature_C', 'Humidity',
    'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh',
    'Field_Area_hectare', 'Previous_Irrigation_mm',
    # Engineered features
    'Moisture_x_Temp', 'Rainfall_x_Humidity',
    'Temp_x_Humidity', 'Moisture_x_Rainfall',
    'Irrigation_per_Hectare', 'Rainfall_per_Sunlight',
    'Moisture_Deficit'
]

feature_cols = num_cols + cat_cols

# Build ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ],
    remainder='drop'
)

# Encode target
le_target = LabelEncoder()
y_all = le_target.fit_transform(train_fe[TARGET])
print('Target classes:', le_target.classes_)
print('Encoded as   :', np.unique(y_all))

# Extract feature DataFrames (not yet transformed — pipeline will handle it)
X_all_df = train_fe[feature_cols]
X_test_df = test_fe[feature_cols]
test_ids = test_fe['id'].values

print(f'\nX_all shape  : {X_all_df.shape}')
print(f'X_test shape : {X_test_df.shape}')
print(f'y_all shape  : {y_all.shape}')

---
## 6. Train / Validation Split

In [ ]:
X_train_df, X_val_df, y_train, y_val = train_test_split(
    X_all_df, y_all,
    test_size=0.15,
    random_state=SEED,
    stratify=y_all
)

print(f'Train size : {X_train_df.shape[0]:,}')
print(f'Val size   : {X_val_df.shape[0]:,}')
print()
print('Train class distribution:')
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(le_target.classes_, counts):
    print(f'  {cls}: {cnt:,} ({cnt/len(y_train)*100:.1f}%)')

---
## 7. SVM — Stratified K-Fold Cross-Validation (Baseline)

We first establish a baseline SVM with default parameters using 5-fold cross-validation on a stratified subsample.

**Why subsample?** RBF SVM has O(n²)–O(n³) time complexity. Training on 630K rows would take days. Subsampling to ~100K is standard professional practice for SVM on large datasets.

In [ ]:
# Stratified subsample for SVM tuning
rng = np.random.default_rng(SEED)
tune_size = min(SVM_TUNE_SAMPLE, len(X_train_df))

# Stratified subsample: maintain class proportions
from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, train_size=tune_size, random_state=SEED)
tune_idx, _ = next(sss.split(X_train_df, y_train))

X_tune_df = X_train_df.iloc[tune_idx]
y_tune    = y_train[tune_idx]

print(f'Tuning subsample: {len(X_tune_df):,} rows')
unique, counts = np.unique(y_tune, return_counts=True)
for cls, cnt in zip(le_target.classes_, counts):
    print(f'  {cls}: {cnt:,} ({cnt/len(y_tune)*100:.1f}%)')

# --- Baseline SVM with cross-validation ---
print('\n--- Baseline SVM (RBF, C=1.0, class_weight=balanced) ---')
print('Running 5-fold Stratified CV...')

svm_baseline_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('svm', SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        class_weight='balanced',
        decision_function_shape='ovr',
        random_state=SEED,
        cache_size=1000
    ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_results = cross_validate(
    svm_baseline_pipe, X_tune_df, y_tune,
    cv=cv,
    scoring=['accuracy', 'f1_macro'],
    return_train_score=True,
    n_jobs=-1
)

print(f'\nCV Accuracy : {cv_results["test_accuracy"].mean():.4f} ± {cv_results["test_accuracy"].std():.4f}')
print(f'CV F1 Macro : {cv_results["test_f1_macro"].mean():.4f} ± {cv_results["test_f1_macro"].std():.4f}')
print(f'Train Acc   : {cv_results["train_accuracy"].mean():.4f} ± {cv_results["train_accuracy"].std():.4f}')
print(f'Train F1    : {cv_results["train_f1_macro"].mean():.4f} ± {cv_results["train_f1_macro"].std():.4f}')

print('\nPer-fold results:')
for i in range(5):
    print(f'  Fold {i+1}: Acc={cv_results["test_accuracy"][i]:.4f}, F1={cv_results["test_f1_macro"][i]:.4f}')

---
## 8. Hyperparameter Tuning with GridSearchCV

We tune the SVM hyperparameters `C` and `gamma` using grid search with stratified 3-fold cross-validation.

- **C**: Regularization parameter — controls the trade-off between margin width and misclassification
- **gamma**: RBF kernel coefficient — controls the reach of each training example
- **Scoring**: `f1_macro` — appropriate for imbalanced multiclass (gives equal weight to all classes)

In [ ]:
svm_tune_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('svm', SVC(
        kernel='rbf',
        class_weight='balanced',
        decision_function_shape='ovr',
        random_state=SEED,
        cache_size=1000
    ))
])

param_grid = {
    'svm__C':     [0.1, 1.0, 10.0],
    'svm__gamma': ['scale', 'auto', 0.01, 0.1]
}

cv_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

print(f'GridSearchCV: {len(param_grid["svm__C"]) * len(param_grid["svm__gamma"])} parameter combinations × 3 folds')
print(f'Training on {len(X_tune_df):,} samples...')
print('This may take 15-30 minutes. Please wait...\n')

grid_search = GridSearchCV(
    svm_tune_pipe,
    param_grid,
    cv=cv_tune,
    scoring='f1_macro',
    refit=True,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_search.fit(X_tune_df, y_tune)

print(f'\n{"="*50}')
print(f'Best Parameters : {grid_search.best_params_}')
print(f'Best F1 Macro   : {grid_search.best_score_:.4f}')
print(f'{"="*50}')

In [ ]:
# --- Display full CV results table ---
cv_df = pd.DataFrame(grid_search.cv_results_)
cv_display = cv_df[[
    'param_svm__C', 'param_svm__gamma',
    'mean_test_score', 'std_test_score',
    'mean_train_score', 'rank_test_score'
]].sort_values('rank_test_score')

cv_display.columns = ['C', 'gamma', 'Mean F1 (test)', 'Std F1', 'Mean F1 (train)', 'Rank']
print('Grid Search Results (sorted by rank):')
print(cv_display.to_string(index=False))

# Check for overfitting
best_idx = grid_search.best_index_
train_f1 = cv_df.loc[best_idx, 'mean_train_score']
test_f1  = cv_df.loc[best_idx, 'mean_test_score']
gap = train_f1 - test_f1
print(f'\nOverfit check: Train F1={train_f1:.4f}, Test F1={test_f1:.4f}, Gap={gap:.4f}')
if gap > 0.05:
    print('⚠ Potential overfitting detected (gap > 0.05)')
else:
    print('✓ No significant overfitting')

---
## 9. Train Final SVM on Larger Subsample

Now that we have the best hyperparameters, we train the final SVM on a larger subsample for better generalization.

In [ ]:
best_C     = grid_search.best_params_['svm__C']
best_gamma = grid_search.best_params_['svm__gamma']

print(f'Training final SVM with C={best_C}, gamma={best_gamma}')

# Larger stratified subsample for final training
final_size = min(SVM_TRAIN_SAMPLE, len(X_train_df))
sss_final = StratifiedShuffleSplit(n_splits=1, train_size=final_size, random_state=SEED)
final_idx, _ = next(sss_final.split(X_train_df, y_train))

X_final_df = X_train_df.iloc[final_idx]
y_final    = y_train[final_idx]

print(f'Final training sample: {len(X_final_df):,} rows')

# Build final pipeline
svm_final_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('svm', SVC(
        kernel='rbf',
        C=best_C,
        gamma=best_gamma,
        class_weight='balanced',
        decision_function_shape='ovr',
        probability=True,         # Enable for ROC curves & ensemble
        random_state=SEED,
        cache_size=1000
    ))
])

print('Fitting final SVM...')
svm_final_pipe.fit(X_final_df, y_final)
print('Training complete.')

# Evaluate on held-out validation set
svm_val_preds = svm_final_pipe.predict(X_val_df)
svm_val_acc   = accuracy_score(y_val, svm_val_preds)
svm_val_f1    = f1_score(y_val, svm_val_preds, average='macro')

print(f'\nValidation Accuracy : {svm_val_acc:.4f}')
print(f'Validation F1 Macro : {svm_val_f1:.4f}')

---
## 10. Detailed SVM Evaluation

In [ ]:
# --- 10a. Classification Report ---
print('='*60)
print('CLASSIFICATION REPORT — SVM (Tuned)')
print('='*60)
print(classification_report(
    y_val, svm_val_preds,
    target_names=le_target.classes_,
    digits=4
))

In [ ]:
# --- 10b. Confusion Matrix ---
cm = confusion_matrix(y_val, svm_val_preds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_,
            ax=axes[0])
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('Actual', fontsize=12)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')

# Normalized (recall per class)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_,
            ax=axes[1])
axes[1].set_xlabel('Predicted', fontsize=12)
axes[1].set_ylabel('Actual', fontsize=12)
axes[1].set_title('Confusion Matrix (Normalized by True Class)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# --- 10c. Per-Class ROC Curves (One-vs-Rest) ---
svm_val_proba = svm_final_pipe.predict_proba(X_val_df)
y_val_bin = label_binarize(y_val, classes=np.arange(len(le_target.classes_)))

fig, ax = plt.subplots(figsize=(8, 6))
colors_roc = ['#2196F3', '#FF9800', '#F44336']

for i, (cls, color) in enumerate(zip(le_target.classes_, colors_roc)):
    fpr, tpr, _ = roc_curve(y_val_bin[:, i], svm_val_proba[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{cls} (AUC = {roc_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Per-Class ROC Curves — SVM (One-vs-Rest)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 11. Baseline Comparisons

We compare our tuned SVM against Random Forest and LightGBM baselines to contextualize its performance.

In [ ]:
# --- 11a. Random Forest Baseline ---
print('Training Random Forest baseline...')
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight='balanced',
    n_jobs=-1,
    random_state=SEED
)

# RF doesn't need scaling but we use preprocessor for consistent encoding
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', rf_model)
])

rf_pipe.fit(X_train_df, y_train)  # RF can handle full data
rf_val_preds = rf_pipe.predict(X_val_df)
rf_val_acc = accuracy_score(y_val, rf_val_preds)
rf_val_f1  = f1_score(y_val, rf_val_preds, average='macro')
print(f'RF Validation Accuracy : {rf_val_acc:.4f}')
print(f'RF Validation F1 Macro : {rf_val_f1:.4f}')
print()
print(classification_report(y_val, rf_val_preds, target_names=le_target.classes_, digits=4))

In [ ]:
# --- 11b. LightGBM Baseline ---
print('Training LightGBM baseline...')
lgbm_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)

lgbm_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('lgbm', lgbm_model)
])

lgbm_pipe.fit(X_train_df, y_train)
lgbm_val_preds = lgbm_pipe.predict(X_val_df)
lgbm_val_acc = accuracy_score(y_val, lgbm_val_preds)
lgbm_val_f1  = f1_score(y_val, lgbm_val_preds, average='macro')
print(f'LGBM Validation Accuracy : {lgbm_val_acc:.4f}')
print(f'LGBM Validation F1 Macro : {lgbm_val_f1:.4f}')
print()
print(classification_report(y_val, lgbm_val_preds, target_names=le_target.classes_, digits=4))

In [ ]:
# --- 11c. Model Comparison Visualization ---
results = pd.DataFrame({
    'Model': ['SVM (Tuned RBF)', 'Random Forest', 'LightGBM'],
    'Accuracy': [svm_val_acc, rf_val_acc, lgbm_val_acc],
    'F1 Macro': [svm_val_f1, rf_val_f1, lgbm_val_f1]
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bar_colors = ['#F44336', '#2196F3', '#4CAF50']

# Accuracy comparison
bars1 = axes[0].bar(results['Model'], results['Accuracy'], color=bar_colors, edgecolor='black', linewidth=0.5)
axes[0].set_ylim(min(results['Accuracy']) - 0.03, max(results['Accuracy']) + 0.03)
axes[0].set_ylabel('Validation Accuracy', fontsize=12)
axes[0].set_title('Model Comparison — Accuracy', fontsize=13, fontweight='bold')
for bar, val in zip(bars1, results['Accuracy']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')

# F1 Macro comparison
bars2 = axes[1].bar(results['Model'], results['F1 Macro'], color=bar_colors, edgecolor='black', linewidth=0.5)
axes[1].set_ylim(min(results['F1 Macro']) - 0.03, max(results['F1 Macro']) + 0.03)
axes[1].set_ylabel('Validation F1 Macro', fontsize=12)
axes[1].set_title('Model Comparison — F1 Macro Score', fontsize=13, fontweight='bold')
for bar, val in zip(bars2, results['F1 Macro']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print('\nFull comparison table:')
print(results.to_string(index=False))
print(f'\nBest model by F1 Macro: {results.loc[results["F1 Macro"].idxmax(), "Model"]}')

---
## 12. Retrain SVM on Full Training Data & Generate Submission

We retrain the tuned SVM on the largest feasible subsample, then generate predictions on the test set.

In [ ]:
# Retrain on larger subsample from ALL training data (not just train split)
FINAL_SAMPLE = min(SVM_TRAIN_SAMPLE, len(X_all_df))
sss_submit = StratifiedShuffleSplit(n_splits=1, train_size=FINAL_SAMPLE, random_state=SEED)
submit_idx, _ = next(sss_submit.split(X_all_df, y_all))

X_submit_df = X_all_df.iloc[submit_idx]
y_submit    = y_all[submit_idx]

print(f'Retraining on {len(X_submit_df):,} samples with best params (C={best_C}, gamma={best_gamma})...')

svm_submit_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('svm', SVC(
        kernel='rbf',
        C=best_C,
        gamma=best_gamma,
        class_weight='balanced',
        decision_function_shape='ovr',
        probability=False,    # Not needed for final predictions
        random_state=SEED,
        cache_size=1000
    ))
])

svm_submit_pipe.fit(X_submit_df, y_submit)
print('Training complete.')

# Predict on test set
print('Generating test predictions...')
test_preds_encoded = svm_submit_pipe.predict(X_test_df)
test_preds_labels  = le_target.inverse_transform(test_preds_encoded)

# Create submission
submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': test_preds_labels
})

submission.to_csv('submission.csv', index=False)
print(f'\nSubmission saved to submission.csv')
print(f'Shape: {submission.shape}')
print()
print('Prediction distribution:')
print(submission['Irrigation_Need'].value_counts())
print()
submission.head(10)

---
## Summary

| Step | Description |
|------|-------------|
| **Preprocessing** | `ColumnTransformer` with `OneHotEncoder` (categoricals) + `StandardScaler` (numerics) |
| **Feature Engineering** | 7 interaction/ratio features (moisture×temp, rainfall×humidity, etc.) |
| **Class Imbalance** | `class_weight='balanced'` — adjusts C inversely proportional to class frequency |
| **Model** | SVM with RBF kernel, tuned via `GridSearchCV` |
| **Tuning** | `C` ∈ {0.1, 1, 10}, `gamma` ∈ {scale, auto, 0.01, 0.1}, scored by `f1_macro` |
| **Validation** | Stratified 5-fold CV + held-out 15% validation set |
| **Evaluation** | Classification report, confusion matrix (raw + normalized), per-class ROC curves |
| **Baselines** | Random Forest (300 trees), LightGBM (500 rounds) for comparison |